# Drug RAG — 청킹 + 임베딩 + pgvector 적재

**실행 전 체크리스트**
- [ ] 런타임 유형 변경 → T4 GPU
- [ ] Secrets 등록: AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, AWS_DEFAULT_REGION, S3_BUCKET

**흐름**: Colab PostgreSQL 실행 → Silver S3 → 청킹 → 임베딩 → pgvector 적재 → pg_dump → S3 저장

In [ ]:
# Cell 1: PostgreSQL + pgvector 설치
!apt-get -qq update
!apt-get -qq install -y postgresql postgresql-contrib
!apt-get -qq install -y postgresql-15-pgvector || echo 'apt pgvector 없음, 소스 빌드 시도'
import subprocess
r = subprocess.run(['sudo', '-u', 'postgres', 'psql', '-c', 'CREATE EXTENSION IF NOT EXISTS vector;'], capture_output=True, text=True)
if r.returncode != 0:
    print('소스 빌드 시작...')
    !cd /tmp && git clone --branch v0.7.0 https://github.com/pgvector/pgvector.git 2>/dev/null
    !cd /tmp/pgvector && make -s && make install -s
print('설치 완료')

In [ ]:
# Cell 2: PostgreSQL 시작 + DB 생성
import time
!service postgresql start
time.sleep(3)
!sudo -u postgres psql -c "CREATE DATABASE drug_rag;"
!sudo -u postgres psql -d drug_rag -c "CREATE EXTENSION IF NOT EXISTS vector;"
!sudo -u postgres psql -c "ALTER USER postgres PASSWORD 'drugrag123';"
print('PostgreSQL 준비 완료 / DB: drug_rag / PW: drugrag123')

In [ ]:
# Cell 3: Python 패키지 설치
!pip install -q boto3 sentence-transformers psycopg2-binary pgvector langchain-text-splitters tqdm
print('패키지 설치 완료')

In [ ]:
# Cell 4: 환경변수 로드
import os
from google.colab import userdata

os.environ['AWS_ACCESS_KEY_ID']     = userdata.get('AWS_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('AWS_SECRET_ACCESS_KEY')
os.environ['AWS_DEFAULT_REGION']    = userdata.get('AWS_DEFAULT_REGION')
S3_BUCKET = userdata.get('S3_BUCKET')

PG_HOST = 'localhost'
PG_PORT = '5432'
PG_DB   = 'drug_rag'
PG_USER = 'postgres'
PG_PASS = 'drugrag123'

print(f'S3 버킷: {S3_BUCKET}')

In [ ]:
# Cell 5: GPU 확인
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'디바이스: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('GPU 없음. 런타임 유형을 T4 GPU로 변경하세요.')

In [ ]:
# Cell 6: 임베딩 모델 로드
from sentence_transformers import SentenceTransformer
MODEL_NAME = 'intfloat/multilingual-e5-large'
print(f'로딩 중: {MODEL_NAME}')
model = SentenceTransformer(MODEL_NAME, device=device)
EMBED_DIM = model.get_sentence_embedding_dimension()
print(f'완료 / 차원: {EMBED_DIM}')

In [ ]:
# Cell 7: S3 유틸 + PostgreSQL 연결
import json, boto3, psycopg2
from psycopg2.extras import execute_values

s3 = boto3.client('s3', region_name=os.environ['AWS_DEFAULT_REGION'])

def list_s3_keys(prefix):
    paginator = s3.get_paginator('list_objects_v2')
    keys = []
    for page in paginator.paginate(Bucket=S3_BUCKET, Prefix=prefix):
        for obj in page.get('Contents', []):
            if obj['Key'].endswith('.json'):
                keys.append(obj['Key'])
    return keys

def download_json(key):
    res = s3.get_object(Bucket=S3_BUCKET, Key=key)
    return json.loads(res['Body'].read().decode('utf-8'))

conn = psycopg2.connect(host=PG_HOST, port=PG_PORT, dbname=PG_DB, user=PG_USER, password=PG_PASS)
conn.autocommit = False
print('PostgreSQL 연결 완료')

In [ ]:
# Cell 8: 테이블 생성
with conn.cursor() as cur:
    cur.execute(f'''
        CREATE TABLE IF NOT EXISTS drug_documents (
            id           SERIAL PRIMARY KEY,
            doc_id       TEXT UNIQUE NOT NULL,
            doc_type     TEXT NOT NULL,
            chunk_index  INTEGER DEFAULT 0,
            text         TEXT NOT NULL,
            embedding    vector({EMBED_DIM}),
            name         TEXT,
            source_name  TEXT,
            source_url   TEXT,
            license      TEXT,
            provider     TEXT,
            item_url     TEXT,
            manufacturer TEXT,
            image_url    TEXT,
            dur_type     TEXT,
            ingredient_a TEXT,
            ingredient_b TEXT,
            metadata     JSONB,
            created_at   TIMESTAMP DEFAULT NOW()
        );
    ''')
    cur.execute('CREATE INDEX IF NOT EXISTS drug_docs_type_idx ON drug_documents (doc_type);')
conn.commit()
print('테이블 생성 완료')

In [ ]:
# Cell 9: 청킹 함수
from langchain_text_splitters import RecursiveCharacterTextSplitter

CHUNK_CONFIG = {
    'drug_info':       {'size': 512, 'overlap': 64},
    'dur_interaction': {'size': 256, 'overlap': 0},
    'symptom_disease': {'size': 512, 'overlap': 64},
}

def chunk_document(doc):
    doc_type = doc.get('doc_type', 'drug_info')
    cfg = CHUNK_CONFIG.get(doc_type, {'size': 512, 'overlap': 64})
    text = doc.get('text', '')
    if not text:
        return []
    if doc_type == 'dur_interaction' or len(text) <= cfg['size']:
        return [{'chunk_index': 0, 'text': text}]
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=cfg['size'], chunk_overlap=cfg['overlap'], separators=['|', ' ', '']
    )
    return [{'chunk_index': i, 'text': c} for i, c in enumerate(splitter.split_text(text))]

print('청킹 함수 준비 완료')

In [ ]:
# Cell 10: 임베딩 + 적재 함수
import numpy as np
from tqdm import tqdm

def embed_texts(texts, batch_size=64):
    passages = [f'passage: {t}' for t in texts]
    return model.encode(passages, batch_size=batch_size, show_progress_bar=False, normalize_embeddings=True)

def upsert_chunks(docs, batch_size=64):
    rows = []
    for doc in docs:
        chunks = chunk_document(doc)
        meta = doc.get('metadata', {})
        for chunk in chunks:
            rows.append({
                'doc_id':       f"{doc['doc_id']}_c{chunk['chunk_index']}",
                'doc_type':     doc.get('doc_type', ''),
                'chunk_index':  chunk['chunk_index'],
                'text':         chunk['text'],
                'name':         meta.get('name', ''),
                'source_name':  meta.get('source_name', ''),
                'source_url':   meta.get('source_url', ''),
                'license':      meta.get('license', ''),
                'provider':     meta.get('provider', ''),
                'item_url':     meta.get('item_url', ''),
                'manufacturer': meta.get('manufacturer', ''),
                'image_url':    meta.get('image_url', ''),
                'dur_type':     meta.get('dur_type', ''),
                'ingredient_a': meta.get('ingredient_a', ''),
                'ingredient_b': meta.get('ingredient_b', ''),
                'metadata':     json.dumps(meta, ensure_ascii=False),
            })
    if not rows:
        return 0
    embeddings = embed_texts([r['text'] for r in rows], batch_size=batch_size)
    with conn.cursor() as cur:
        execute_values(cur, '''
            INSERT INTO drug_documents (
                doc_id, doc_type, chunk_index, text, embedding,
                name, source_name, source_url, license, provider,
                item_url, manufacturer, image_url,
                dur_type, ingredient_a, ingredient_b, metadata
            ) VALUES %s
            ON CONFLICT (doc_id) DO UPDATE SET
                text=EXCLUDED.text, embedding=EXCLUDED.embedding, metadata=EXCLUDED.metadata
        ''', [
            (r['doc_id'], r['doc_type'], r['chunk_index'], r['text'],
             embeddings[i].tolist(),
             r['name'], r['source_name'], r['source_url'], r['license'], r['provider'],
             r['item_url'], r['manufacturer'], r['image_url'],
             r['dur_type'], r['ingredient_a'], r['ingredient_b'], r['metadata'])
            for i, r in enumerate(rows)
        ])
    conn.commit()
    return len(rows)

print('임베딩/적재 함수 준비 완료')

In [ ]:
# Cell 11: Silver 전체 처리
SILVER_PREFIXES = [
    'silver/drugs/public_drug/',
    'silver/drugs/health_kr_drug/',
    'silver/dur_interactions/combination_ban/',
    'silver/dur_interactions/pregnancy_ban/',
    'silver/dur_interactions/elderly_caution/',
    'silver/dur_interactions/dose_caution/',
    'silver/dur_interactions/effect_duplicate/',
    'silver/dur_interactions/age_specific_ban/',
    'silver/dur_interactions/dosing_period_caution/',
    'silver/symptoms/health_portal/',
    'silver/symptoms/health_kr_disease/',
]

total_saved = 0
for prefix in SILVER_PREFIXES:
    keys = list_s3_keys(prefix)
    if not keys:
        print(f'스킵: {prefix}')
        continue
    label = prefix.split('/')[-2]
    print(f'\n▶ {label} ({len(keys)}개 파일)')
    for key in tqdm(keys, desc=label):
        try:
            data = download_json(key)
            items = data.get('items', [])
            for i in range(0, len(items), 200):
                total_saved += upsert_chunks(items[i:i+200])
        except Exception as e:
            print(f'오류 {key}: {str(e)[:80]}')

print(f'\n전체 완료: {total_saved:,}개 청크 적재')

In [ ]:
# Cell 12: 적재 결과 확인
with conn.cursor() as cur:
    cur.execute('SELECT doc_type, COUNT(*) FROM drug_documents GROUP BY doc_type ORDER BY COUNT(*) DESC')
    rows = cur.fetchall()
total = 0
for doc_type, cnt in rows:
    print(f'  {doc_type:<25} {cnt:>8,}건')
    total += cnt
print(f'  합계{"":21} {total:>8,}건')

In [ ]:
# Cell 13: 벡터 인덱스 생성 (적재 완료 후)
print('벡터 인덱스 생성 중...')
with conn.cursor() as cur:
    cur.execute('''
        CREATE INDEX IF NOT EXISTS drug_docs_embedding_idx
        ON drug_documents USING ivfflat (embedding vector_cosine_ops) WITH (lists = 100);
    ''')
conn.commit()
print('벡터 인덱스 생성 완료')

In [ ]:
# Cell 14: 검색 테스트
def search(query, doc_type=None, top_k=5):
    q_emb = model.encode(f'query: {query}', normalize_embeddings=True).tolist()
    with conn.cursor() as cur:
        if doc_type:
            cur.execute('''
                SELECT name, source_name, text, 1-(embedding <=> %s::vector) AS score
                FROM drug_documents WHERE doc_type=%s
                ORDER BY embedding <=> %s::vector LIMIT %s
            ''', (q_emb, doc_type, q_emb, top_k))
        else:
            cur.execute('''
                SELECT name, source_name, text, 1-(embedding <=> %s::vector) AS score
                FROM drug_documents
                ORDER BY embedding <=> %s::vector LIMIT %s
            ''', (q_emb, q_emb, top_k))
        return cur.fetchall()

print('=== 두통에 먹는 약 ===')
for name, source, text, score in search('두통에 먹는 약', doc_type='drug_info'):
    print(f'[{score:.3f}] {name} ({source})')
    print(f'  {text[:100]}')

print('\n=== 이부프로펜 병용금기 ===')
for name, source, text, score in search('이부프로펜 병용금기', doc_type='dur_interaction'):
    print(f'[{score:.3f}] {text[:120]}')

In [ ]:
# Cell 15: pg_dump → S3 저장 (세션 종료 전 필수!)
import subprocess
from datetime import datetime

dump_path = '/tmp/drug_rag_pgvector.dump'
s3_key = f'gold/pgvector_dump/{datetime.now().strftime("%Y-%m-%d")}/drug_rag_pgvector.dump'

print('pg_dump 실행 중...')
r = subprocess.run(['sudo', '-u', 'postgres', 'pg_dump', '-Fc', '-d', 'drug_rag', '-f', dump_path],
                   capture_output=True, text=True)
if r.returncode == 0:
    size_mb = os.path.getsize(dump_path) / 1024 / 1024
    print(f'dump 크기: {size_mb:.1f} MB')
    s3.upload_file(dump_path, S3_BUCKET, s3_key)
    print(f'S3 저장 완료: s3://{S3_BUCKET}/{s3_key}')
else:
    print(f'오류: {r.stderr}')

## Cell 16: 로컬 restore 명령어

Colab 세션 종료 후 로컬 CMD에서 실행:

```cmd
aws s3 cp s3://drug-rag-datalake-331145994962/gold/pgvector_dump/YYYY-MM-DD/drug_rag_pgvector.dump .\drug_rag_pgvector.dump

docker exec -it drug_rag_postgres psql -U airflow -c "CREATE DATABASE drug_rag;"

docker cp drug_rag_pgvector.dump drug_rag_postgres:/tmp/

docker exec -it drug_rag_postgres pg_restore -U airflow -d drug_rag /tmp/drug_rag_pgvector.dump
```